In [ ]:
import os
WORKSHOP_RESOURCE_GROUP = "YOUR_RESOURCE_GROUP"
WORKSHOP_AUTH_MODE = "managed-identity"
os.environ["WORKSHOP_RESOURCE_GROUP"] = WORKSHOP_RESOURCE_GROUP
os.environ["WORKSHOP_AUTH_MODE"] = WORKSHOP_AUTH_MODE
os.environ["CHAT_DEPLOYMENT_NAME"] = ""
os.environ["CHAT_MODEL_NAME"] = ""
os.environ["EMBEDDING_DEPLOYMENT_NAME"] = ""
os.environ["EMBEDDING_MODEL_NAME"] = ""
os.environ["KB_MCP_ENDPOINT"] = ""
os.environ["RESOURCE_GROUP_NAME"] = WORKSHOP_RESOURCE_GROUP
os.environ["FOUNDRY_PROJECT_NAME"] = ""

In [ ]:
import os
import shlex
import subprocess

resource_group_name = WORKSHOP_RESOURCE_GROUP
auth_mode = WORKSHOP_AUTH_MODE

cmd = [
    "bash",
    "../../scripts/assign-workshop-env.sh",
    "--resource-group",
    resource_group_name,
    "--auth-mode",
    auth_mode,
]

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stderr.strip():
    print(result.stderr.strip())

for line in result.stdout.splitlines():
    line = line.strip()
    if not line.startswith("export "):
        continue
    key, raw_value = line[len("export "):].split("=", 1)
    parsed = shlex.split(raw_value)
    os.environ[key] = parsed[0] if parsed else ""

print("Workshop environment variables loaded into notebook kernel. Azure token auth uses AzureCliCredential via az login.")

In [ ]:
import importlib
import sys
from pathlib import Path

NOTEBOOK_PATH_CANDIDATES = [Path.cwd(), Path.cwd() / "AgentWorkshop" / "Notebook"]
for candidate in NOTEBOOK_PATH_CANDIDATES:
    if (candidate / "workshop_bootstrap.py").exists():
        resolved_candidate = str(candidate.resolve())
        if resolved_candidate not in sys.path:
            sys.path.insert(0, resolved_candidate)

import workshop_bootstrap
importlib.reload(workshop_bootstrap)
build_workshop_config = workshop_bootstrap.build_workshop_config

CONFIG_OVERRIDES = {
    "resource_group_name": "",
    "location": "",
    "subscription_id": "",
    "foundry_account_name": "",
    "foundry_project_name": "",
    "foundry_project_endpoint": "",
    "foundry_project_api_key": "",
    "search_service_name": "",
    "search_api_key": "",
    "storage_account_name": "",
    "application_insights_name": "",
    "model_zone": "",
}

config = build_workshop_config(CONFIG_OVERRIDES)
config.show()

# Workshop 5: Multi-Agent Group Chat

This notebook mirrors docs/multi-agent-2.md and creates:
- Orchestrator-Agent
- Data-Analyst-Agent
- Research-Agent
- Recipe-Agent
- Insights-Agent

It also writes a group-chat workflow YAML artifact for the Foundry workflow designer.

In [ ]:
# Uncomment this cell in a clean kernel.
# %pip install --quiet "azure-ai-projects>=2.0.0" azure-identity

In [ ]:
import os
from pathlib import Path

from azure.ai.projects.models import AutoCodeInterpreterToolParam, CodeInterpreterTool, PromptAgentDefinition

from workshop_bootstrap import build_project_client, write_text

if not config.foundry_project_endpoint:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT or provide foundry_project_endpoint in CONFIG_OVERRIDES.")

CHAT_DEPLOYMENT_NAME = os.getenv("CHAT_DEPLOYMENT_NAME", "").strip()
if not CHAT_DEPLOYMENT_NAME:
    raise ValueError("Set CHAT_DEPLOYMENT_NAME in your environment.")

DATASET_GENERAL = Path("../../data/Coffee/CoffeeCSV/GeneralHealth/synthetic_mental_health_dataset.csv").resolve()
DATASET_LARGE = Path("../../data/Coffee/CoffeeCSV/mentalHealth/synthetic_coffee_health_10000.csv").resolve()

project = build_project_client(config)
openai = project.get_openai_client()

with DATASET_GENERAL.open("rb") as file_handle:
    general_file = openai.files.create(purpose="assistants", file=file_handle)
with DATASET_LARGE.open("rb") as file_handle:
    large_file = openai.files.create(purpose="assistants", file=file_handle)

print("Uploaded files for Data-Analyst-Agent:", general_file.id, large_file.id)

In [ ]:
ORCHESTRATOR_AGENT_NAME = "Orchestrator-Agent"
DATA_ANALYST_AGENT_NAME = "Data-Analyst-Agent"
RESEARCH_AGENT_NAME = "Research-Agent"
RECIPE_AGENT_NAME = "Recipe-Agent"
INSIGHTS_AGENT_NAME = "Insights-Agent"
WORKFLOW_NAME = "coffee-group-chat"
WORKFLOW_FILE = Path("../Agent/coffee-group-chat.workflow.yaml").resolve()

orchestrator_prompt = """
You are the Orchestrator Agent.

Your responsibilities:
* Understand user intent
* Route requests to appropriate agents:
    * Data --> Data-Analyst-Agent
    * Research --> Research-Agent
    * Recipes --> Recipe-Agent

* Coordinate multi-agent collaboration
* Ensure final answer is produced by Insights-Agent

ALWAYS return your responses in JSON with one of the following agent names.
    * Data-Analyst-Agent
    * Research-Agent
    * Recipe-Agent
    * Insights-Agent

JSON Schema
{
  "type": "object",
  "properties": {
    "next-agent": {
      "type": "string"
    },
    "next-agent-request": {
      "type": "string"
    },
    "history": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "agent": {
            "type": "string"
          },
          "output": {
            "type": "string"
          }
        },
        "required": [
          "agent",
          "output"
        ]
      }
    }
  },
  "required": [
    "next-agent",
    "next-agent-request",
    "history"
  ]
}

for example (Always Compress the JSON), 
{"next-agent":"NAME OF NEXT AGENT","next-agent-request":"NEXT AGENT REQUEST TEXT","history":[{"agent":"NAME OF AGENT","output":"TEXT RESULT OF AGENT"},{"agent":"NAME OF AGENT","output":"TEXT RESULT OF AGENT"},{"agent":"NAME OF AGENT","output":"TEXT RESULT OF AGENT"}]}

Do NOT answer directly. Always delegate.
"""

data_analyst_prompt = """
You are the Data Analyst Agent.

You analyze CSV datasets using Code Interpreter.

Your tasks:
* Perform statistical analysis
* Identify trends between coffee consumption and health metrics
* Generate insights based on structured data

Return structured summaries for other agents.
"""

research_prompt = """
You are the Research Agent.

You retrieve evidence from the knowledge base.

Your tasks:
* Provide scientific insights on coffee and health
* Focus on validated research findings
* Summarize clearly for downstream synthesis
"""

recipe_prompt = """
You are the Recipe Agent.

You provide coffee recipes using the knowledge base.

Your tasks:
    * Suggest recipes aligned with health goals
    * Include preparation steps
    * Consider insights from other agents
"""

insights_prompt = """
You are the Insights Agent.

Your responsibilities:
* Combine outputs from all agents
* Produce a final, cohesive answer
* Ensure clarity and actionability

Do NOT introduce new information. Only synthesize existing outputs.
"""

project.agents.create_version(
    agent_name=ORCHESTRATOR_AGENT_NAME,
    definition=PromptAgentDefinition(model=CHAT_DEPLOYMENT_NAME, instructions=orchestrator_prompt, tools=[]),
    description="Routes requests among specialist agents.",
)

project.agents.create_version(
    agent_name=DATA_ANALYST_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_DEPLOYMENT_NAME,
        instructions=data_analyst_prompt,
        tools=[
            CodeInterpreterTool(
                container=AutoCodeInterpreterToolParam(file_ids=[general_file.id, large_file.id])
            )
        ],
    ),
    description="Performs quantitative analysis on workshop CSV files.",
)

project.agents.create_version(
    agent_name=RESEARCH_AGENT_NAME,
    definition=PromptAgentDefinition(model=CHAT_DEPLOYMENT_NAME, instructions=research_prompt, tools=[]),
    description="Provides evidence-backed health research context.",
)

project.agents.create_version(
    agent_name=RECIPE_AGENT_NAME,
    definition=PromptAgentDefinition(model=CHAT_DEPLOYMENT_NAME, instructions=recipe_prompt, tools=[]),
    description="Provides coffee recipe and preparation context.",
)

project.agents.create_version(
    agent_name=INSIGHTS_AGENT_NAME,
    definition=PromptAgentDefinition(model=CHAT_DEPLOYMENT_NAME, instructions=insights_prompt, tools=[]),
    description="Synthesizes specialist outputs into final response.",
)

print("Created five group-chat agents.")

In [ ]:
workflow_yaml = f"""
kind: workflow
name: {WORKFLOW_NAME}
description: Group chat multi-agent workflow for coffee workshop
trigger:
  kind: OnConversationStart
  id: trigger_wf
  actions:
    - kind: SetVariable
      id: init_history
      variable: Local.ConvoHistory
      value: =System.LastMessage.Text
    - kind: InvokeAzureAgent
      id: orchestrator_step
      agent:
        name: {ORCHESTRATOR_AGENT_NAME}
      conversationId: =System.ConversationId
      input:
        messages: =Local.ConvoHistory
      output:
        autoSend: true
        responseObject: Local.OrchestratorOutput
        messages: Local.LastOrchestratorMessage
    - kind: SetVariable
      id: next_agent_name
      variable: Local.NextAgentName
      value: =Upper(Trim(Text(Local.OrchestratorOutput.'next-agent')))
    - kind: ConditionGroup
      id: route_group
      conditions:
        - id: route_data_agent
          condition: =Local.NextAgentName = "DATA-ANALYST-AGENT"
          actions:
            - kind: InvokeAzureAgent
              id: data_agent_step
              agent:
                name: {DATA_ANALYST_AGENT_NAME}
              conversationId: =System.ConversationId
              input:
                messages: =Local.LastOrchestratorMessage
              output:
                autoSend: true
                messages: Local.ConvoHistory
            - kind: GotoAction
              id: goto_orchestrator_from_data
              actionId: orchestrator_step
        - id: route_research_agent
          condition: =Local.NextAgentName = "RESEARCH-AGENT"
          actions:
            - kind: InvokeAzureAgent
              id: research_agent_step
              agent:
                name: {RESEARCH_AGENT_NAME}
              conversationId: =System.ConversationId
              input:
                messages: =Local.LastOrchestratorMessage
              output:
                autoSend: true
                messages: Local.ConvoHistory
            - kind: GotoAction
              id: goto_orchestrator_from_research
              actionId: orchestrator_step
        - id: route_recipe_agent
          condition: =Local.NextAgentName = "RECIPE-AGENT"
          actions:
            - kind: InvokeAzureAgent
              id: recipe_agent_step
              agent:
                name: {RECIPE_AGENT_NAME}
              conversationId: =System.ConversationId
              input:
                messages: =Local.LastOrchestratorMessage
              output:
                autoSend: true
                messages: Local.ConvoHistory
            - kind: GotoAction
              id: goto_orchestrator_from_recipe
              actionId: orchestrator_step
      elseActions:
        - kind: InvokeAzureAgent
          id: insights_step
          agent:
            name: {INSIGHTS_AGENT_NAME}
          conversationId: =System.ConversationId
          input:
            messages: =Local.LastOrchestratorMessage
          output:
            autoSend: true
    - kind: EndConversation
      id: end_conversation
""".strip()

write_text(WORKFLOW_FILE, workflow_yaml)
print(f"Workflow YAML saved to: {WORKFLOW_FILE}")
print("Try the Prompt:")
print("Give me a healthy coffee recipe and explain to me what the research says it will do to my body")

## Suggested Prompts

- What trends exist between coffee consumption and mental health?
- What does research say about coffee and anxiety?
- Suggest a coffee recipe aligned with reducing stress.